# Chapter 05a — Eigenvalues & Eigenvectors

A square matrix $A$ acts on a vector by **transforming** it — rotating,
stretching, shearing. For *most* directions the output points somewhere new.
But some special directions are only **stretched** (or shrunk, or flipped),
never rotated. Those directions are the **eigenvectors**, and the stretch
factor is the **eigenvalue**.

$$A\,v = \lambda\, v, \qquad v \neq 0.$$

This single equation runs through the rest of machine learning: it is the heart
of diagonalization, the SVD, and PCA. Run each cell with **Shift + Enter**.

## 1. Setup

We use NumPy for the linear algebra and Matplotlib for pictures. A fixed random
generator (`default_rng(0)`) keeps results reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # reproducible randomness
np.set_printoptions(precision=4, suppress=True)  # tidy number printing

## 2. The defining equation $A v = \lambda v$

Let's take a concrete matrix and a vector that happens to be an eigenvector,
and check the equation by hand.

$$A = \begin{pmatrix} 2 & 0 \\ 0 & 3 \end{pmatrix}.$$

A diagonal matrix simply scales the $x$-axis by 2 and the $y$-axis by 3, so the
axis directions are obvious eigenvectors.

In [ ]:
A = np.array([[2.0, 0.0],
              [0.0, 3.0]])

v = np.array([1.0, 0.0])   # the x-axis direction
print("A @ v =", A @ v)    # -> [2, 0] = 2 * v, so lambda = 2
print("2 * v =", 2 * v)

The output of `A @ v` is exactly `2 * v`: the vector kept its direction and was
scaled by $\lambda = 2$. That is what "eigenvector" means.

## 3. Computing eigenvalues with `np.linalg.eig`

For anything bigger than a diagonal matrix we let NumPy do the work.
`np.linalg.eig(A)` returns

- `vals` — a 1-D array of eigenvalues $\lambda_i$,
- `vecs` — a matrix whose **columns** are the matching eigenvectors (each
  normalized to length 1).

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])

vals, vecs = np.linalg.eig(A)
print("eigenvalues :", vals)
print("eigenvectors (columns):")
print(vecs)

**Reading the output:** eigenvalue `vals[i]` goes with the eigenvector in
column `vecs[:, i]`. Let's verify $A v = \lambda v$ for the first pair.

In [ ]:
lam = vals[0]
v = vecs[:, 0]            # the i-th eigenvector is a COLUMN

print("A @ v     =", A @ v)
print("lam * v   =", lam * v)
print("close?    ", np.allclose(A @ v, lam * v))   # True

## 4. Seeing it: eigen vs non-eigen directions

A picture makes the idea click. We draw a vector (blue) and its image $A v$
(red). For an **eigenvector** the red arrow lies on the same line as the blue
one — only the length changes. For a generic direction the red arrow swings off
to the side.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eig(A)

# Pick an eigenvector and a deliberately non-eigen direction.
eig_dir = vecs[:, 0]                    # an eigenvector
non_dir = np.array([1.0, 0.0])          # the x-axis: NOT an eigenvector here

def draw(ax, v, title):
    Av = A @ v
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
              color='tab:blue', label='v')
    ax.quiver(0, 0, Av[0], Av[1], angles='xy', scale_units='xy', scale=1,
              color='tab:red', label='A v')
    ax.set_xlim(-1, 4); ax.set_ylim(-1, 4)
    ax.set_aspect('equal'); ax.grid(True)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(title); ax.legend(loc='upper left')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
draw(ax1, eig_dir, "Eigenvector: A v stays on the line")
draw(ax2, non_dir, "Non-eigenvector: A v rotates away")
plt.tight_layout()
plt.show()

On the left the red arrow is a stretched copy of the blue one — same line. On
the right the red arrow points in a new direction: the $x$-axis is not an
eigenvector of this $A$.

## 5. Diagonalization $A = P D P^{-1}$

Collect the eigenvectors as the columns of $P$ and the eigenvalues on the
diagonal of $D$. Then

$$A = P\,D\,P^{-1}.$$

This says: to apply $A$, first change coordinates into the eigenvector basis
($P^{-1}$), scale each axis by its eigenvalue ($D$), then change back ($P$).
Powers become easy too: $A^k = P\,D^k\,P^{-1}$.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eig(A)

P = vecs                 # columns are eigenvectors
D = np.diag(vals)        # eigenvalues on the diagonal
Pinv = np.linalg.inv(P)

reconstructed = P @ D @ Pinv
print("P D P^-1 =")
print(reconstructed)
print("matches A?", np.allclose(reconstructed, A))   # True

In [ ]:
# Bonus: A^3 the easy way, via D^3
A3_fast = P @ np.diag(vals**3) @ Pinv
A3_slow = A @ A @ A
print("same A^3?", np.allclose(A3_fast, A3_slow))     # True

## 6. Symmetric matrices are special: use `np.linalg.eigh`

A **symmetric** matrix ($A = A^{\top}$) always has

- **real** eigenvalues, and
- eigenvectors that are mutually **orthogonal** (perpendicular).

For these, prefer `np.linalg.eigh` ("h" for Hermitian/symmetric): it is faster,
more accurate, and returns the eigenvalues in ascending order.

In [ ]:
# Build a guaranteed-symmetric matrix: M + M^T is always symmetric.
M = rng.standard_normal((3, 3))
A = M + M.T
print("symmetric?", np.allclose(A, A.T))   # True

vals, vecs = np.linalg.eigh(A)
print("eigenvalues (real, ascending):", vals)

In [ ]:
# Orthogonal eigenvectors: V^T V should be the identity matrix.
print("V^T V =")
print(vecs.T @ vecs)        # ~ identity -> columns are orthonormal

Because the eigenvectors are orthonormal, $P^{-1} = P^{\top}$, and
diagonalization simplifies to $A = V\,D\,V^{\top}$.

In [ ]:
V, D = vecs, np.diag(vals)
print("V D V^T matches A?", np.allclose(V @ D @ V.T, A))   # True

---
## ✍️ Exercise 1 (solution included)

For the matrix
$$A = \begin{pmatrix} 4 & 1 \\ 2 & 3 \end{pmatrix},$$
compute its eigenvalues and eigenvectors with `np.linalg.eig`, then verify
numerically that $A v = \lambda v$ for **each** eigenpair.

**Solution:**

In [ ]:
A = np.array([[4.0, 1.0],
              [2.0, 3.0]])
vals, vecs = np.linalg.eig(A)

for i in range(len(vals)):
    lam = vals[i]
    v = vecs[:, i]
    print(f"lambda = {lam:.4f},  A v = {A @ v},  lam v = {lam * v}")
    print("   match?", np.allclose(A @ v, lam * v))

## ✍️ Exercise 2 (solution included)

The **trace** (sum of the diagonal) of a matrix equals the **sum** of its
eigenvalues, and the **determinant** equals their **product**. Verify both
facts for a random $4\times 4$ matrix.

**Solution:**

In [ ]:
A = rng.standard_normal((4, 4))
vals, _ = np.linalg.eig(A)

print("trace        :", np.trace(A))
print("sum of eigs  :", np.sum(vals).real)     # .real drops tiny imaginary dust
print()
print("determinant  :", np.linalg.det(A))
print("prod of eigs :", np.prod(vals).real)

---
## 📝 Homework (no solutions provided)

1. Take $A = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ (a 90° rotation).
   Compute its eigenvalues with `np.linalg.eig`. Why are they **complex**, and
   what does that say about real eigenvector directions for a pure rotation?
2. Build a random symmetric $5\times 5$ matrix. Use `np.linalg.eigh` and confirm
   that every eigenvalue is real and that the eigenvectors are orthonormal
   (check `V.T @ V`).
3. For the symmetric matrix in (2), reconstruct it as $V D V^{\top}$ and report
   the maximum absolute reconstruction error.
4. Write a function `power_iteration(A, steps=100)` that starts from a random
   vector, repeatedly applies `A` and renormalizes, and returns the dominant
   eigenvector. Compare its result to the top eigenvector from `np.linalg.eig`.